# Assignment 28: Q&A RAG Chatbot  with Message History
---

## Task 1: Load Documents

In [54]:
from langchain_community.document_loaders import PyPDFLoader

In [55]:
loader = PyPDFLoader("genai_document.pdf")

documents = loader.load()

In [56]:
print("Number of documents:", len(documents))

Number of documents: 1


In [57]:
documents[0].page_content[:1000]

'Dummy GenAI Knowledge Document\nGenerative AI is a branch of artificial intelligence that can create new content such as text, code, images, and\nsummaries. Large Language Models (LLMs) generate text by predicting likely tokens from the context\nprovided to them.\nRetrieval-Augmented Generation (RAG) improves an LLM application by retrieving relevant information from\nexternal documents before generating an answer. A typical RAG pipeline loads documents, splits them into\nchunks, creates embeddings, stores the embeddings in a vector store, retrieves relevant chunks, and sends\nthe retrieved context to the language model.\nPrompt templates make LLM applications easier to maintain. Instead of hard-coding every question, a\ntemplate can contain placeholders such as {question}. The application fills those placeholders dynamically at\nruntime.\nChat prompt templates are useful for conversational applications because they separate system instructions,\nhuman messages, and optional AI messag

## Task 2: Text Splitting

In [58]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [59]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [60]:
chunks = text_splitter.split_documents(documents)

In [61]:
len(chunks)

3

In [62]:
chunks[0].page_content

'Dummy GenAI Knowledge Document\nGenerative AI is a branch of artificial intelligence that can create new content such as text, code, images, and\nsummaries. Large Language Models (LLMs) generate text by predicting likely tokens from the context\nprovided to them.\nRetrieval-Augmented Generation (RAG) improves an LLM application by retrieving relevant information from\nexternal documents before generating an answer. A typical RAG pipeline loads documents, splits them into'

In [63]:
chunks[0].metadata

{'producer': 'ReportLab PDF Library - (opensource)',
 'creator': '(unspecified)',
 'creationdate': '2026-09-13T11:47:10+00:00',
 'author': '(anonymous)',
 'keywords': '',
 'moddate': '2026-09-13T11:47:10+00:00',
 'subject': '(unspecified)',
 'title': '(anonymous)',
 'trapped': '/False',
 'source': 'genai_document.pdf',
 'total_pages': 1,
 'page': 0,
 'page_label': '1'}

# PART 2 — Vector Store & Retriever
## Task 3: Create Embeddings

In [64]:
from langchain_huggingface import HuggingFaceEmbeddings

In [65]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5170.90it/s]


In [66]:
sample_embedding = embeddings.embed_query(
    "What is machine learning?"
)

In [67]:
len(sample_embedding)

384

## Task 4: Store Embeddings in Vector Store

In [68]:
from langchain_community.vectorstores import FAISS

In [26]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

In [27]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [28]:
query = "What is Generative AI?"

results = retriever.invoke(query)



In [29]:
for i, doc in enumerate(results):
    print(f"\n- Result {i + 1} -")
    print(doc.page_content)


- Result 1 -
Dummy GenAI Knowledge Document
Generative AI is a branch of artificial intelligence that can create new content such as text, code, images, and
summaries. Large Language Models (LLMs) generate text by predicting likely tokens from the context
provided to them.
Retrieval-Augmented Generation (RAG) improves an LLM application by retrieving relevant information from
external documents before generating an answer. A typical RAG pipeline loads documents, splits them into

- Result 2 -
human messages, and optional AI messages. This structure makes the intended conversation flow explicit.
Example rule: Answer questions only from the supplied context. If the answer is not present in the context,
say that the information is not available in the provided document.

- Result 3 -
chunks, creates embeddings, stores the embeddings in a vector store, retrieves relevant chunks, and sends
the retrieved context to the language model.
Prompt templates make LLM applications easier to maintai

# PART 3 — Prompt with Message History
## Task 5: RAG Prompt Template

In [36]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [37]:
load_dotenv()

True

In [38]:
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful question-answering assistant.

Answer the user's question using ONLY the provided context.

If the answer cannot be found in the context, say:
"I don't know."

Do not use outside knowledge.

Context:
{context}
"""
    ),
    
    MessagesPlaceholder(variable_name="chat_history"),
    
    ("human", "{question}")
])

In [39]:
context = "\n\n".join(
    doc.page_content for doc in results
)

chat_history = []

prompt_value = rag_prompt.invoke({
    "context": context,
    "chat_history": chat_history,
    "question": query
})

print(prompt_value)

messages=[SystemMessage(content='You are a helpful question-answering assistant.\n\nAnswer the user\'s question using ONLY the provided context.\n\nIf the answer cannot be found in the context, say:\n"I don\'t know."\n\nDo not use outside knowledge.\n\nContext:\nDummy GenAI Knowledge Document\nGenerative AI is a branch of artificial intelligence that can create new content such as text, code, images, and\nsummaries. Large Language Models (LLMs) generate text by predicting likely tokens from the context\nprovided to them.\nRetrieval-Augmented Generation (RAG) improves an LLM application by retrieving relevant information from\nexternal documents before generating an answer. A typical RAG pipeline loads documents, splits them into\n\nhuman messages, and optional AI messages. This structure makes the intended conversation flow explicit.\nExample rule: Answer questions only from the supplied context. If the answer is not present in the context,\nsay that the information is not available in

In [40]:
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)

In [43]:
chat_history = []

def ask_question(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    messages = rag_prompt.invoke({
        "context": context,
        "chat_history": chat_history,
        "question": question
    })

    response = llm.invoke(messages)

    chat_history.append({
        "role": "human",
        "content": question
    })

    chat_history.append({
        "role": "ai",
        "content": response.content
    })

    return response.content

In [44]:
ask_question("What is GenAI?")

'Generative AI is a branch of artificial intelligence that can create new content such as text, code, images, and summaries.'

In [ ]:
ask_question("Can you give me an example?")

In [ ]:
ask_question("What are its main applications?")

# PART 4 — Q&A RAG Chain with Message History
## Task 6: Build RAG Chain

In [45]:
def rag_chat(question):
    
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )
    messages = rag_prompt.invoke({
        "context": context,
        "chat_history": chat_history,
        "question": question
    })
    response = llm.invoke(messages)
    return response.content

## Task 7: Maintain Message History

In [49]:
from langchain_core.messages import HumanMessage, AIMessage

In [50]:
chat_history = []

def rag_chat(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )
    messages = rag_prompt.invoke({
        "context": context,
        "chat_history": chat_history,
        "question": question
    })

    response = llm.invoke(messages)
    chat_history.append(
        HumanMessage(content=question)
    )
    chat_history.append(
        AIMessage(content=response.content)
    )

    return response.content

In [ ]:
rag_chat("What is GenAI?")

'Generative AI is a branch of artificial intelligence that can create new content such as text, code, images, and summaries.'

In [ ]:
rag_chat("Can you give me an example?")

In [ ]:
rag_chat("What are its main applications?")

## Task 8: Trimming Chat History

In [52]:
MAX_MESSAGES = 6

def trim_history(history):
    if len(history) > MAX_MESSAGES:
        return history[-MAX_MESSAGES:]
    
    return history

In [53]:
chat_history = []

def rag_chat(question):
    chat_history[:] = trim_history(chat_history)
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )
    messages = rag_prompt.invoke({
        "context": context,
        "chat_history": chat_history,
        "question": question
    })
    response = llm.invoke(messages)
    chat_history.append(
        HumanMessage(content=question)
    )
    chat_history.append(
        AIMessage(content=response.content)
    )
    chat_history[:] = trim_history(chat_history)

    return response.content

In [ ]:
rag_chat("What is GenAI?")

'Generative AI is a branch of artificial intelligence that can create new content such as text, code, images, and summaries.'

In [ ]:
rag_chat("Can you give me an example?")

In [ ]:
rag_chat("What are its main applications?")

In [ ]:
for message in chat_history:
    print(
        type(message).__name__,
        ":",
        message.content[:200]
    )

# PART 5 — Testing the Conversational RAG Bot
## Task 9: Multi-Turn Q&A Testing

In [ ]:
rag_chat("What is GenAI?")

'Generative AI is a branch of artificial intelligence that can create new content such as text, code, images, and summaries.'

In [ ]:
rag_chat("Can you give me an example?")

In [ ]:
rag_chat("What are its main applications?")

In [ ]:
for message in chat_history:
    print(type(message).__name__, ":", message.content)

# PART 6 — Mini Project: Conversational RAG Assistant
## Task 10: Build Final Chatbot

In [ ]:
chat_history = []

MAX_MESSAGES = 6

In [ ]:
def trim_history(history):
    if len(history) > MAX_MESSAGES:
        return history[-MAX_MESSAGES:]
    return history

In [ ]:
def conversational_rag(question):
    chat_history[:] = trim_history(chat_history)
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )
    messages = rag_prompt.invoke({
        "context": context,
        "chat_history": chat_history,
        "question": question
    })
    response = llm.invoke(messages)
    chat_history.append(
        HumanMessage(content=question)
    )

    chat_history.append(
        AIMessage(content=response.content)
    )

    chat_history[:] = trim_history(chat_history)

    return response.content

In [ ]:
conversational_rag("What is GenAI?")

'Generative AI is a branch of artificial intelligence that can create new content such as text, code, images, and summaries.'

In [ ]:
conversational_rag("Can you give me an example?")

In [ ]:
conversational_rag("What are its main applications?")

In [ ]:
while True:
    question = input("You: ")

    if question.lower() == "exit":
        print("Chatbot: Goodbye!")
        break

    answer = conversational_rag(question)

    print("Chatbot:", answer)

# Task 11: Observations & Insights

1. Difference between normal RAG and conversational RAG
- Normal RAG retrieves relevant documents for each user question and generates an answer from the retrieved context.
- Conversational RAG combines document retrieval with conversation history. Therefore, it can understand follow-up questions that depend on previous messages.

2. Role of message history in follow-up questions
- Message history provides the previous conversation to the LLM. This allows the chatbot to understand references

3. Trade-offs between long memory and performance
- Longer history provides more conversational context, but it also increases:
    - Token usage
    - Processing time
    - API cost
    - Prompt size
- Shorter history reduces these costs but can remove information needed for older references.